# Percobaan 8 - Poisson Bivariate + Outcome Classifier

Notebook ini lanjut dari Percobaan 7.

Combo yang dicoba:
- reconstructed historical features,
- CatBoost MAE/RMSE regression ensemble,
- CatBoost outcome classifier,
- CatBoost Poisson regressors untuk lambda gol,
- bivariate Poisson score distribution dengan shared lambda,
- empirical score prior,
- expected AW-MAE score selector.

Output submission tidak menimpa Percobaan 6/7.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict, deque
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, CatBoostClassifier

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

## 1. Path dan konfigurasi

Default `TASK_TYPE='CPU'` supaya stabil. Kalau CatBoost GPU di kernel `py_gpu_ready` aman, boleh ganti ke `GPU`.

In [2]:
BASE_PATH = Path.home() / "Downloads" / "Gammafest"
DATA_PATH = BASE_PATH / "dataset"
OUTPUT_DIR = BASE_PATH / "experiments" / "percobaan 8 - poisson bivariate"

TRAIN_PATH = DATA_PATH / "train.csv"
TEST_PATH = DATA_PATH / "test.csv"
SAMPLE_PATH = DATA_PATH / "sample submission.csv"

SUBMISSION_PATH = OUTPUT_DIR / "submission_poisson_bivariate_jalur_a.csv"
SUBMISSION_ROUNDCLIP_PATH = OUTPUT_DIR / "submission_poisson_bivariate_roundclip.csv"
VALID_REPORT_PATH = OUTPUT_DIR / "validation_poisson_bivariate_regression_report.csv"
POSTPROCESS_REPORT_PATH = OUTPUT_DIR / "postprocess_poisson_bivariate_report.csv"

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = "GPU"
MAX_SCORE = 6
N_RANDOM_BLENDS = 2000

ELO_INIT = 1500.0
ELO_K_DEFAULT = 20
ELO_K_IMPORTANT = 40
IMPORTANT_TOURNAMENTS = {
    "FIFA World Cup", "AFC Asian Cup", "AFC Championship", "UEFA Euro",
    "Africa Cup of Nations", "African Cup of Nations", "Copa America", "Copa América",
    "Gold Cup", "CONCACAF Gold Cup"
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

Output: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 8 - poisson bivariate


## 2. Load data

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

train_raw["date_dt"] = pd.to_datetime(train_raw["date"], errors="coerce")
test_raw["date_dt"] = pd.to_datetime(test_raw["date"], errors="coerce")

print("train:", train_raw.shape, train_raw["date_dt"].min(), "->", train_raw["date_dt"].max())
print("test :", test_raw.shape, test_raw["date_dt"].min(), "->", test_raw["date_dt"].max())
print("sample:", sample.shape)
display(train_raw.head(2))
display(test_raw.head(2))

train: (78772, 48) 1872-11-30 00:00:00 -> 2011-08-04 00:00:00
test : (42422, 21) 2011-08-06 00:00:00 -> 2026-03-31 00:00:00
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169,2011-08-06
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169,2011-08-06


## 3. Static features, temporal split, dan AW-MAE constants

In [4]:
CAT_COLS = [
    "gender", "team", "opponent", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
]

STATIC_NUM_COLS = [
    "is_home", "neutral",
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue", "distance_travel_team", "distance_travel_opp", "temperature_venue",
]

DATE_FEATURES = ["year", "month", "dayofweek"]

TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Championship": 1.80,
    "AFC Asian Cup": 1.80,
    "UEFA Euro": 1.80,
    "Copa America": 1.80,
    "Copa América": 1.80,
    "Africa Cup of Nations": 1.80,
    "African Cup of Nations": 1.80,
    "Gold Cup": 1.75,
    "CONCACAF Gold Cup": 1.75,
    "FIFA World Cup qualification": 1.50,
    "UEFA Euro qualification": 1.40,
    "AFC Asian Cup qualification": 1.40,
    "Friendly": 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def add_basic_features(df):
    df = df.copy()
    dt = pd.to_datetime(df["date"], errors="coerce")
    df["date_dt"] = dt
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["dayofweek"] = dt.dt.dayofweek
    df["tournament_weight"] = df["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT)
    return df


match_dates = train_raw.groupby("match_id")["date_dt"].min().sort_values()
split_idx = int(len(match_dates) * (1 - VALID_FRAC))
train_match_ids = set(match_dates.index[:split_idx])
val_match_ids = set(match_dates.index[split_idx:])

tr_raw = train_raw[train_raw["match_id"].isin(train_match_ids)].copy()
val_raw = train_raw[train_raw["match_id"].isin(val_match_ids)].copy()

print("Train fold rows:", tr_raw.shape, tr_raw["date_dt"].min(), "->", tr_raw["date_dt"].max())
print("Valid fold rows:", val_raw.shape, val_raw["date_dt"].min(), "->", val_raw["date_dt"].max())
print("Train matches:", len(train_match_ids), "Valid matches:", len(val_match_ids))

Train fold rows: (63016, 48) 1872-11-30 00:00:00 -> 2005-01-30 00:00:00
Valid fold rows: (15756, 48) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00
Train matches: 31508 Valid matches: 7878


## 4. Forward-only reconstruction

`can_update_outcome=True` hanya untuk history train yang boleh memperbarui state. Validation/test tidak memperbarui form, H2H, atau Elo dengan target mereka.
Schedule tetap memperbarui last-match date karena tanggal pertandingan memang diketahui.

In [5]:
def points_from_score(gf, ga):
    if gf > ga:
        return 3
    if gf == ga:
        return 1
    return 0


def elo_k(tournament):
    return ELO_K_IMPORTANT if tournament in IMPORTANT_TOURNAMENTS else ELO_K_DEFAULT


def summarize_form(history):
    if len(history) == 0:
        return {
            "points_last5": np.nan,
            "points_last10": np.nan,
            "gd_last5": np.nan,
            "avg_goals_last5": np.nan,
            "avg_conceded_last5": np.nan,
            "win_rate_last10": np.nan,
            "matches_known": 0,
        }
    last5 = list(history)[-5:]
    last10 = list(history)[-10:]
    return {
        "points_last5": float(sum(x["points"] for x in last5)),
        "points_last10": float(sum(x["points"] for x in last10)),
        "gd_last5": float(sum(x["gd"] for x in last5)),
        "avg_goals_last5": float(np.mean([x["gf"] for x in last5])),
        "avg_conceded_last5": float(np.mean([x["ga"] for x in last5])),
        "win_rate_last10": float(np.mean([x["points"] == 3 for x in last10])),
        "matches_known": int(len(history)),
    }


def build_reconstructed_features(input_df):
    df = add_basic_features(input_df).copy()
    df["_orig_order"] = np.arange(len(df))
    if "can_update_outcome" not in df.columns:
        df["can_update_outcome"] = df["team_goals"].notna() if "team_goals" in df.columns else False
    if "split_name" not in df.columns:
        df["split_name"] = "unknown"

    for col in ["team_goals", "opp_goals", "rank_team", "rank_opponent"]:
        if col not in df.columns:
            df[col] = np.nan

    df = df.sort_values(["date_dt", "match_id", "Id"]).reset_index(drop=True)

    form = defaultdict(lambda: deque(maxlen=50))
    h2h = defaultdict(lambda: deque(maxlen=20))
    elo = defaultdict(lambda: ELO_INIT)
    last_match_date = {}
    latest_rank = {}
    feature_rows = []

    for match_id, grp in df.groupby("match_id", sort=False):
        match_date = grp["date_dt"].iloc[0]

        for _, row in grp.iterrows():
            gender = row["gender"]
            team = row["team"]
            opp = row["opponent"]
            team_key = (gender, team)
            opp_key = (gender, opp)
            pair_key = (gender, team, opp)

            tf = summarize_form(form[team_key])
            of = summarize_form(form[opp_key])
            hhist = list(h2h[pair_key])[-5:]

            h2h_points = float(sum(x["points"] for x in hhist)) if hhist else np.nan
            h2h_gd = float(sum(x["gd"] for x in hhist)) if hhist else np.nan
            h2h_matches = int(len(hhist))

            team_last_date = last_match_date.get(team_key)
            opp_last_date = last_match_date.get(opp_key)
            days_team = (match_date - team_last_date).days if team_last_date is not None and pd.notna(match_date) else np.nan
            days_opp = (match_date - opp_last_date).days if opp_last_date is not None and pd.notna(match_date) else np.nan

            rank_team_latest = latest_rank.get(team_key, np.nan)
            rank_opp_latest = latest_rank.get(opp_key, np.nan)

            feature_rows.append({
                "_orig_order": row["_orig_order"],
                "team_points_last5_recon": tf["points_last5"],
                "opp_points_last5_recon": of["points_last5"],
                "points_last5_diff_recon": tf["points_last5"] - of["points_last5"] if pd.notna(tf["points_last5"]) and pd.notna(of["points_last5"]) else np.nan,
                "team_points_last10_recon": tf["points_last10"],
                "opp_points_last10_recon": of["points_last10"],
                "points_last10_diff_recon": tf["points_last10"] - of["points_last10"] if pd.notna(tf["points_last10"]) and pd.notna(of["points_last10"]) else np.nan,
                "team_gd_last5_recon": tf["gd_last5"],
                "opp_gd_last5_recon": of["gd_last5"],
                "gd_last5_diff_recon": tf["gd_last5"] - of["gd_last5"] if pd.notna(tf["gd_last5"]) and pd.notna(of["gd_last5"]) else np.nan,
                "team_avg_goals_last5_recon": tf["avg_goals_last5"],
                "team_avg_conceded_last5_recon": tf["avg_conceded_last5"],
                "opp_avg_goals_last5_recon": of["avg_goals_last5"],
                "opp_avg_conceded_last5_recon": of["avg_conceded_last5"],
                "avg_goals_diff_recon": tf["avg_goals_last5"] - of["avg_goals_last5"] if pd.notna(tf["avg_goals_last5"]) and pd.notna(of["avg_goals_last5"]) else np.nan,
                "avg_conceded_diff_recon": tf["avg_conceded_last5"] - of["avg_conceded_last5"] if pd.notna(tf["avg_conceded_last5"]) and pd.notna(of["avg_conceded_last5"]) else np.nan,
                "team_win_rate_last10_recon": tf["win_rate_last10"],
                "opp_win_rate_last10_recon": of["win_rate_last10"],
                "win_rate_diff_recon": tf["win_rate_last10"] - of["win_rate_last10"] if pd.notna(tf["win_rate_last10"]) and pd.notna(of["win_rate_last10"]) else np.nan,
                "h2h_points_last5_recon": h2h_points,
                "h2h_gd_last5_recon": h2h_gd,
                "h2h_matches_last5_recon": h2h_matches,
                "days_since_last_match_team_recon": days_team,
                "days_since_last_match_opp_recon": days_opp,
                "days_since_last_match_diff_recon": days_team - days_opp if pd.notna(days_team) and pd.notna(days_opp) else np.nan,
                "elo_team_recon": float(elo[team_key]),
                "elo_opponent_recon": float(elo[opp_key]),
                "elo_diff_recon": float(elo[team_key] - elo[opp_key]),
                "rank_team_latest_recon": rank_team_latest,
                "rank_opponent_latest_recon": rank_opp_latest,
                "rank_diff_latest_recon": rank_team_latest - rank_opp_latest if pd.notna(rank_team_latest) and pd.notna(rank_opp_latest) else np.nan,
                "rank_missing_team_recon": int(pd.isna(rank_team_latest)),
                "rank_missing_opp_recon": int(pd.isna(rank_opp_latest)),
                "team_matches_known_recon": tf["matches_known"],
                "opp_matches_known_recon": of["matches_known"],
                "matches_known_diff_recon": tf["matches_known"] - of["matches_known"],
            })

        for _, row in grp.iterrows():
            last_match_date[(row["gender"], row["team"])] = match_date

        can_update = bool(grp["can_update_outcome"].fillna(False).all())
        has_scores = grp["team_goals"].notna().all() and grp["opp_goals"].notna().all()
        if can_update and has_scores:
            for _, row in grp.iterrows():
                gender = row["gender"]
                team = row["team"]
                opp = row["opponent"]
                gf = float(row["team_goals"])
                ga = float(row["opp_goals"])
                pts = points_from_score(gf, ga)
                gd = gf - ga
                team_key = (gender, team)
                form[team_key].append({"points": pts, "gf": gf, "ga": ga, "gd": gd})
                h2h[(gender, team, opp)].append({"points": pts, "gd": gd})

                if pd.notna(row.get("rank_team", np.nan)):
                    latest_rank[team_key] = float(row["rank_team"])
                if pd.notna(row.get("rank_opponent", np.nan)):
                    latest_rank[(gender, opp)] = float(row["rank_opponent"])

            row = grp.iloc[0]
            gender = row["gender"]
            team_key = (gender, row["team"])
            opp_key = (gender, row["opponent"])
            e_team = elo[team_key]
            e_opp = elo[opp_key]
            expected_team = 1 / (1 + 10 ** ((e_opp - e_team) / 400))
            gf = float(row["team_goals"])
            ga = float(row["opp_goals"])
            actual_team = 1.0 if gf > ga else 0.5 if gf == ga else 0.0
            k = elo_k(row["tournament"])
            elo[team_key] = e_team + k * (actual_team - expected_team)
            elo[opp_key] = e_opp + k * ((1 - actual_team) - (1 - expected_team))

    feature_df = pd.DataFrame(feature_rows)
    return df.merge(feature_df, on="_orig_order", how="left").sort_values("_orig_order").reset_index(drop=True)

## 5. Build features untuk eval dan final test

In [6]:
tr_for_eval = tr_raw.copy()
tr_for_eval["split_name"] = "train_fold"
tr_for_eval["can_update_outcome"] = True

val_for_eval = val_raw.copy()
val_for_eval["split_name"] = "valid_fold"
val_for_eval["can_update_outcome"] = False

eval_input = pd.concat([tr_for_eval, val_for_eval], ignore_index=True, sort=False)
print("Building eval reconstructed features...")
eval_recon = build_reconstructed_features(eval_input)
tr_recon = eval_recon[eval_recon["split_name"] == "train_fold"].copy()
val_recon = eval_recon[eval_recon["split_name"] == "valid_fold"].copy()

full_train_for_final = train_raw.copy()
full_train_for_final["split_name"] = "full_train"
full_train_for_final["can_update_outcome"] = True

test_for_final = test_raw.copy()
test_for_final["split_name"] = "test"
test_for_final["can_update_outcome"] = False
test_for_final["team_goals"] = np.nan
test_for_final["opp_goals"] = np.nan

final_input = pd.concat([full_train_for_final, test_for_final], ignore_index=True, sort=False)
print("Building final reconstructed features...")
final_recon = build_reconstructed_features(final_input)
full_train_recon = final_recon[final_recon["split_name"] == "full_train"].copy()
test_recon = final_recon[final_recon["split_name"] == "test"].copy()

print("tr_recon:", tr_recon.shape)
print("val_recon:", val_recon.shape)
print("full_train_recon:", full_train_recon.shape)
print("test_recon:", test_recon.shape)

Building eval reconstructed features...
Building final reconstructed features...
tr_recon: (63016, 90)
val_recon: (15756, 90)
full_train_recon: (78772, 90)
test_recon: (42422, 90)


## 6. Sanity check dan preprocessing matrix

In [7]:
RECON_FEATURES = [
    "team_points_last5_recon", "opp_points_last5_recon", "points_last5_diff_recon",
    "team_points_last10_recon", "opp_points_last10_recon", "points_last10_diff_recon",
    "team_gd_last5_recon", "opp_gd_last5_recon", "gd_last5_diff_recon",
    "team_avg_goals_last5_recon", "team_avg_conceded_last5_recon",
    "opp_avg_goals_last5_recon", "opp_avg_conceded_last5_recon",
    "avg_goals_diff_recon", "avg_conceded_diff_recon",
    "team_win_rate_last10_recon", "opp_win_rate_last10_recon", "win_rate_diff_recon",
    "h2h_points_last5_recon", "h2h_gd_last5_recon", "h2h_matches_last5_recon",
    "days_since_last_match_team_recon", "days_since_last_match_opp_recon", "days_since_last_match_diff_recon",
    "elo_team_recon", "elo_opponent_recon", "elo_diff_recon",
    "rank_team_latest_recon", "rank_opponent_latest_recon", "rank_diff_latest_recon",
    "rank_missing_team_recon", "rank_missing_opp_recon",
    "team_matches_known_recon", "opp_matches_known_recon", "matches_known_diff_recon",
]

compare_pairs = [
    ("team_points_last5", "team_points_last5_recon"),
    ("opp_points_last5", "opp_points_last5_recon"),
    ("team_points_last10", "team_points_last10_recon"),
    ("opp_points_last10", "opp_points_last10_recon"),
    ("team_gd_last5", "team_gd_last5_recon"),
    ("opp_gd_last5", "opp_gd_last5_recon"),
    ("team_avg_goals_last5", "team_avg_goals_last5_recon"),
    ("team_avg_conceded_last5", "team_avg_conceded_last5_recon"),
    ("opp_avg_goals_last5", "opp_avg_goals_last5_recon"),
    ("opp_avg_conceded_last5", "opp_avg_conceded_last5_recon"),
    ("team_win_rate_last10", "team_win_rate_last10_recon"),
    ("opp_win_rate_last10", "opp_win_rate_last10_recon"),
    ("h2h_points_last5", "h2h_points_last5_recon"),
    ("h2h_gd_last5", "h2h_gd_last5_recon"),
    ("elo_team", "elo_team_recon"),
    ("elo_opponent", "elo_opponent_recon"),
]

rows = []
for orig, recon in compare_pairs:
    if orig in full_train_recon.columns and recon in full_train_recon.columns:
        tmp = full_train_recon[[orig, recon]].dropna()
        corr = tmp[orig].corr(tmp[recon]) if len(tmp) > 2 else np.nan
        rows.append({"original": orig, "reconstructed": recon, "non_null_pairs": len(tmp), "corr": corr})
compare_df = pd.DataFrame(rows).sort_values("corr")
display(compare_df)

BASE_NUM_COLS = STATIC_NUM_COLS + DATE_FEATURES + ["tournament_weight"] + RECON_FEATURES


def finalize_feature_frames(train_df, other_df, fit_name="train"):
    train = train_df.copy()
    other = other_df.copy()

    for df in [train, other]:
        for col in CAT_COLS:
            df[col] = df[col].fillna("Unknown").astype(str)
        for col in BASE_NUM_COLS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col] == -9999, col] = np.nan

    missing_flag_cols = []
    for col in BASE_NUM_COLS:
        flag_col = f"{col}_missing"
        train[flag_col] = train[col].isna().astype(int)
        other[flag_col] = other[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    medians = train[BASE_NUM_COLS].median(numeric_only=True)
    for col in BASE_NUM_COLS:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        other[col] = other[col].fillna(fill_value)

    feature_cols = CAT_COLS + BASE_NUM_COLS + missing_flag_cols
    cat_feature_indices = [feature_cols.index(c) for c in CAT_COLS]
    print(f"{fit_name}: features={len(feature_cols)}, missing train={train[feature_cols].isna().sum().sum()}, missing other={other[feature_cols].isna().sum().sum()}")
    return train, other, feature_cols, cat_feature_indices


tr_model, val_model, FEATURE_COLS, CAT_FEATURE_INDICES = finalize_feature_frames(tr_recon, val_recon, fit_name="eval")
full_train_model, test_model, FINAL_FEATURE_COLS, FINAL_CAT_FEATURE_INDICES = finalize_feature_frames(full_train_recon, test_recon, fit_name="final")

assert FEATURE_COLS == FINAL_FEATURE_COLS
assert CAT_FEATURE_INDICES == FINAL_CAT_FEATURE_INDICES

X_tr = tr_model[FEATURE_COLS]
X_val = val_model[FEATURE_COLS]
y_tr_team = tr_model["team_goals"]
y_tr_opp = tr_model["opp_goals"]
y_val_team = val_model["team_goals"]
y_val_opp = val_model["opp_goals"]

X_full = full_train_model[FEATURE_COLS]
y_full_team = full_train_model["team_goals"]
y_full_opp = full_train_model["opp_goals"]
X_test = test_model[FEATURE_COLS]

print("X_tr:", X_tr.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

,original,reconstructed,non_null_pairs,corr
8,opp_avg_goals_last5,opp_avg_goals_last5_recon,78289,0.800315
5,opp_gd_last5,opp_gd_last5_recon,78289,0.811221
9,opp_avg_conceded_last5,opp_avg_conceded_last5_recon,78289,0.824315
4,team_gd_last5,team_gd_last5_recon,78289,0.839995
1,opp_points_last5,opp_points_last5_recon,78289,0.841340
6,team_avg_goals_last5,team_avg_goals_last5_recon,78289,0.842844
7,team_avg_conceded_last5,team_avg_conceded_last5_recon,78289,0.843728
11,opp_win_rate_last10,opp_win_rate_last10_recon,78289,0.868597
3,opp_points_last10,opp_points_last10_recon,78289,0.875067
0,team_points_last5,team_points_last5_recon,78289,0.890021


eval: features=105, missing train=0, missing other=0
final: features=105, missing train=0, missing other=0
X_tr: (63016, 105) X_val: (15756, 105) X_test: (42422, 105)


## 7. AW-MAE dan post-processing helpers

In [8]:
def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    return np.clip(team, 0, max_score), np.clip(opp, 0, max_score)


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true["team_goals"].to_numpy()
    true_opp = df_true["opp_goals"].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2
    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)
    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    loss = ((mae + penalty) * multiplier) ** 1.5
    weights = df_true["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()
    score = np.sum(loss * weights) / np.sum(weights)
    diag = {
        "AW-MAE": score,
        "MAE_raw_component": mae.mean(),
        "exact_acc": exact.mean(),
        "outcome_acc": outcome.mean(),
        "goal_diff_acc": gd.mean(),
    }
    return score, diag


def evaluate_raw_predictions(name, df_true, team_raw, opp_raw, max_score=MAX_SCORE):
    team_int, opp_int = postprocess_round_clip(team_raw, opp_raw, max_score=max_score)
    score, diag = compute_awmae(df_true, team_int, opp_int)
    return {"model": name, **diag}

## 8. Train CatBoost ensemble

In [9]:
CATBOOST_CONFIGS = [
    {
        "name": "recon_cat_mae_d7_seed42",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1600, learning_rate=0.040, depth=7, l2_leaf_reg=7, random_strength=1.0, bagging_temperature=0.5, random_seed=42),
    },
    {
        "name": "recon_cat_mae_d6_seed7",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1800, learning_rate=0.035, depth=6, l2_leaf_reg=9, random_strength=1.6, bagging_temperature=0.8, random_seed=7),
    },
    {
        "name": "recon_cat_mae_d8_seed99",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1400, learning_rate=0.035, depth=8, l2_leaf_reg=10, random_strength=0.8, bagging_temperature=0.3, random_seed=99),
    },
    {
        "name": "recon_cat_rmse_d7_seed123",
        "params": dict(loss_function="RMSE", eval_metric="MAE", iterations=1500, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.2, bagging_temperature=0.6, random_seed=123),
    },
]

BASE_CAT_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

val_pred_bank = {}
trained_val_models = {}
reports = []

start = time.time()
for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_tr, y_tr_team, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_team), use_best_model=True, early_stopping_rounds=160)
    mo.fit(X_tr, y_tr_opp, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_opp), use_best_model=True, early_stopping_rounds=160)

    pred_t = np.clip(mt.predict(X_val), 0, None)
    pred_o = np.clip(mo.predict(X_val), 0, None)
    val_pred_bank[name] = (pred_t, pred_o)
    trained_val_models[name] = (mt, mo)

    report = evaluate_raw_predictions(name, val_model, pred_t, pred_o, max_score=MAX_SCORE)
    report["best_iter_team"] = mt.best_iteration_
    report["best_iter_opp"] = mo.best_iteration_
    reports.append(report)
    print(report)

print("Training minutes:", (time.time() - start) / 60)
reports_df = pd.DataFrame(reports).sort_values("AW-MAE")
reports_df.to_csv(VALID_REPORT_PATH, index=False)
display(reports_df)
print("Saved:", VALID_REPORT_PATH)


Training recon_cat_mae_d7_seed42


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1747216	test: 1.1324730	best: 1.1324730 (0)	total: 46.6ms	remaining: 1m 14s
150:	learn: 1.0764289	test: 1.0519559	best: 1.0519559 (150)	total: 4.18s	remaining: 40.2s
300:	learn: 1.0436785	test: 1.0319599	best: 1.0319599 (300)	total: 8.43s	remaining: 36.4s
450:	learn: 1.0284859	test: 1.0255014	best: 1.0254976 (449)	total: 12.8s	remaining: 32.6s
600:	learn: 1.0189926	test: 1.0229367	best: 1.0229367 (600)	total: 16.9s	remaining: 28.2s
750:	learn: 1.0123772	test: 1.0220204	best: 1.0220054 (748)	total: 21.1s	remaining: 23.9s
900:	learn: 1.0065709	test: 1.0218252	best: 1.0218103 (899)	total: 25.4s	remaining: 19.7s
1050:	learn: 1.0011882	test: 1.0208099	best: 1.0207939 (1048)	total: 29.1s	remaining: 15.2s
1200:	learn: 0.9961422	test: 1.0202648	best: 1.0202548 (1192)	total: 33.4s	remaining: 11.1s
1350:	learn: 0.9915983	test: 1.0197733	best: 1.0197536 (1339)	total: 37.7s	remaining: 6.95s
1500:	learn: 0.9870839	test: 1.0194230	best: 1.0193861 (1489)	total: 42s	remaining: 2.77s
1599:	

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1747479	test: 1.1325135	best: 1.1325135 (0)	total: 25.9ms	remaining: 41.5s
150:	learn: 1.0770789	test: 1.0521239	best: 1.0521239 (150)	total: 3.44s	remaining: 33s
300:	learn: 1.0444437	test: 1.0329330	best: 1.0329330 (300)	total: 6.86s	remaining: 29.6s
450:	learn: 1.0287079	test: 1.0255316	best: 1.0255316 (450)	total: 11s	remaining: 28s
600:	learn: 1.0197235	test: 1.0229751	best: 1.0229751 (600)	total: 14.9s	remaining: 24.8s
750:	learn: 1.0130423	test: 1.0217808	best: 1.0217697 (749)	total: 18.3s	remaining: 20.7s
900:	learn: 1.0067128	test: 1.0212144	best: 1.0212084 (899)	total: 21.5s	remaining: 16.7s
1050:	learn: 1.0015278	test: 1.0207420	best: 1.0207420 (1050)	total: 24.7s	remaining: 12.9s
1200:	learn: 0.9967525	test: 1.0202303	best: 1.0202228 (1199)	total: 27.6s	remaining: 9.16s
1350:	learn: 0.9921411	test: 1.0199487	best: 1.0199475 (1349)	total: 30.4s	remaining: 5.61s
1500:	learn: 0.9877069	test: 1.0197883	best: 1.0197533 (1493)	total: 33.4s	remaining: 2.2s
1599:	learn:

...

0:	learn: 1.2750192	test: 1.2666925	best: 1.2666925 (0)	total: 29.7ms	remaining: 44.5s
150:	learn: 1.0449129	test: 1.0741839	best: 1.0740738 (148)	total: 4.25s	remaining: 38s
300:	learn: 1.0261660	test: 1.0734326	best: 1.0730031 (254)	total: 8.83s	remaining: 35.2s
450:	learn: 1.0123414	test: 1.0719646	best: 1.0716323 (423)	total: 12.8s	remaining: 29.8s
600:	learn: 1.0007460	test: 1.0696810	best: 1.0696810 (600)	total: 16.4s	remaining: 24.6s
750:	learn: 0.9909190	test: 1.0701147	best: 1.0695063 (700)	total: 20.3s	remaining: 20.2s
900:	learn: 0.9811512	test: 1.0700628	best: 1.0693992 (851)	total: 24.7s	remaining: 16.4s
bestTest = 1.069399157
bestIteration = 851
Shrink model to first 852 iterations.


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2745150	test: 1.2660928	best: 1.2660928 (0)	total: 22.3ms	remaining: 33.5s
150:	learn: 1.0449266	test: 1.0739298	best: 1.0735371 (111)	total: 4.19s	remaining: 37.5s
300:	learn: 1.0258933	test: 1.0733244	best: 1.0729356 (294)	total: 8.77s	remaining: 34.9s
450:	learn: 1.0123885	test: 1.0711365	best: 1.0710530 (448)	total: 13.2s	remaining: 30.6s
600:	learn: 1.0005943	test: 1.0712748	best: 1.0706502 (481)	total: 17.5s	remaining: 26.1s
bestTest = 1.070650169
bestIteration = 481
Shrink model to first 482 iterations.
{'model': 'recon_cat_rmse_d7_seed123', 'AW-MAE': np.float64(3.0884025476117087), 'MAE_raw_component': np.float64(1.0373191165270372), 'exact_acc': np.float64(0.09628078192434628), 'outcome_acc': np.float64(0.5676567656765676), 'goal_diff_acc': np.float64(0.23045189134298044), 'best_iter_team': 851, 'best_iter_opp': 481}
Training minutes: 4.307008862495422


,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
3,recon_cat_rmse_d7_seed123,3.088403,1.037319,0.096281,0.567657,0.230452,851,481
0,recon_cat_mae_d7_seed42,3.109876,0.996954,0.111005,0.512376,0.234704,1585,1599
1,recon_cat_mae_d6_seed7,3.128780,1.000349,0.110498,0.510536,0.234323,1758,1793
2,recon_cat_mae_d8_seed99,3.138308,1.001396,0.109546,0.504252,0.234768,895,971


Saved: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 8 - poisson bivariate\validation_poisson_bivariate_regression_report.csv


## 9. Blend search

In [10]:
model_names = list(val_pred_bank.keys())
team_matrix = np.vstack([val_pred_bank[name][0] for name in model_names])
opp_matrix = np.vstack([val_pred_bank[name][1] for name in model_names])


def score_blend(weights, max_score=MAX_SCORE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred_t = np.average(team_matrix, axis=0, weights=weights)
    pred_o = np.average(opp_matrix, axis=0, weights=weights)
    pred_t_int, pred_o_int = postprocess_round_clip(pred_t, pred_o, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t_int, pred_o_int)
    return score, diag


blend_rows = []
best = {"score": np.inf, "weights": None, "name": None, "diag": None}

for i, name in enumerate(model_names):
    w = np.zeros(len(model_names)); w[i] = 1
    score, diag = score_blend(w)
    blend_rows.append({"blend": f"single_{name}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"single_{name}", "diag": diag}

w = np.ones(len(model_names)) / len(model_names)
score, diag = score_blend(w)
blend_rows.append({"blend": "equal_average", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "equal_average", "diag": diag}

single_scores = []
for i, name in enumerate(model_names):
    w_single = np.zeros(len(model_names)); w_single[i] = 1
    s, _ = score_blend(w_single)
    single_scores.append(s)
w = 1 / np.maximum(np.array(single_scores), 1e-9)
w = w / w.sum()
score, diag = score_blend(w)
blend_rows.append({"blend": "inverse_awmae", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "inverse_awmae", "diag": diag}

rng = np.random.default_rng(RANDOM_STATE)
alpha_options = [0.25, 0.5, 1.0, 2.0]
for k in range(N_RANDOM_BLENDS):
    alpha = alpha_options[k % len(alpha_options)]
    w = rng.dirichlet(np.ones(len(model_names)) * alpha)
    score, diag = score_blend(w)
    if k < 25 or score < best["score"]:
        blend_rows.append({"blend": f"random_{k}_a{alpha}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"random_{k}_a{alpha}", "diag": diag}

blend_df = pd.DataFrame(blend_rows).sort_values("AW-MAE").reset_index(drop=True)
display(blend_df.head(20))

print("Best blend:", best["name"])
print("Best AW-MAE:", best["score"])
print("Diagnostics:", best["diag"])
print("Weights:")
for name, weight in sorted(zip(model_names, best["weights"]), key=lambda x: -x[1]):
    print(f"  {name:<32} {weight:.5f}")

,blend,AW-MAE,w_recon_cat_mae_d7_seed42,w_recon_cat_mae_d6_seed7,w_recon_cat_mae_d8_seed99,w_recon_cat_rmse_d7_seed123
0,random_1156_a0.25,3.064800,0.278117,0.016604,0.127907,0.577372
1,random_118_a1.0,3.065168,0.317943,0.002083,0.113085,0.566889
2,random_31_a2.0,3.068619,0.110881,0.151535,0.142076,0.595508
3,random_5_a0.5,3.071223,0.026982,0.100146,0.299397,0.573474
4,random_9_a0.5,3.073995,0.197650,0.000592,0.028711,0.773046
5,single_recon_cat_rmse_d7_seed123,3.088403,0.000000,0.000000,0.000000,1.000000
6,random_22_a1.0,3.095726,0.398764,0.116891,0.179967,0.304377
7,random_3_a2.0,3.098344,0.392141,0.121147,0.253396,0.233316
8,random_1_a0.5,3.099967,0.017968,0.150514,0.490905,0.340613
9,random_7_a2.0,3.101272,0.428879,0.327866,0.064114,0.179141


Best blend: random_1156_a0.25
Best AW-MAE: 3.064800074036391
Diagnostics: {'AW-MAE': np.float64(3.064800074036391), 'MAE_raw_component': np.float64(1.0101231276973852), 'exact_acc': np.float64(0.10370652449860371), 'outcome_acc': np.float64(0.5422696115765423), 'goal_diff_acc': np.float64(0.23533891850723535)}
Weights:
  recon_cat_rmse_d7_seed123        0.57737
  recon_cat_mae_d7_seed42          0.27812
  recon_cat_mae_d8_seed99          0.12791
  recon_cat_mae_d6_seed7           0.01660


## 10. Outcome classifier win/draw/loss

Regressor menebak jumlah gol, tapi AW-MAE sangat menghukum outcome yang salah.

Cell ini menambahkan beberapa `CatBoostClassifier` untuk memprediksi outcome dari perspektif row:
- `0`: team kalah,
- `1`: seri,
- `2`: team menang.

Probabilitas classifier akan dipakai di post-processing skor integer.

In [11]:
def make_outcome_target(df):
    diff = df["team_goals"].to_numpy() - df["opp_goals"].to_numpy()
    return np.where(diff > 0, 2, np.where(diff < 0, 0, 1)).astype(int)


def aligned_outcome_proba(model, X):
    raw = model.predict_proba(X)
    out = np.zeros((len(X), 3), dtype=float)
    for j, cls in enumerate(model.classes_):
        out[:, int(cls)] = raw[:, j]
    row_sum = out.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return out / row_sum


y_tr_outcome = make_outcome_target(tr_model)
y_val_outcome = make_outcome_target(val_model)
y_full_outcome = make_outcome_target(full_train_model)

CLASSIFIER_CONFIGS = [
    {
        "name": "outcome_cls_d6_seed42",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1200, learning_rate=0.045, depth=6, l2_leaf_reg=6, random_strength=1.0, random_seed=42),
    },
    {
        "name": "outcome_cls_d7_seed7",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1000, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.4, random_seed=7),
    },
    {
        "name": "outcome_cls_d5_seed99",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1400, learning_rate=0.035, depth=5, l2_leaf_reg=5, random_strength=1.8, random_seed=99),
    },
]

BASE_CLS_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

outcome_val_proba_bank = {}
trained_val_classifiers = {}
cls_reports = []

start = time.time()
for cfg in CLASSIFIER_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CLS_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    clf = CatBoostClassifier(**params)
    clf.fit(
        X_tr,
        y_tr_outcome,
        cat_features=CAT_FEATURE_INDICES,
        eval_set=(X_val, y_val_outcome),
        use_best_model=True,
        early_stopping_rounds=140,
    )

    proba = aligned_outcome_proba(clf, X_val)
    pred = proba.argmax(axis=1)
    acc = (pred == y_val_outcome).mean()
    nll = -np.log(np.clip(proba[np.arange(len(y_val_outcome)), y_val_outcome], 1e-12, 1)).mean()

    outcome_val_proba_bank[name] = proba
    trained_val_classifiers[name] = clf
    cls_reports.append({"classifier": name, "outcome_acc": acc, "nll": nll, "best_iter": clf.best_iteration_})
    print(cls_reports[-1])

cls_report_df = pd.DataFrame(cls_reports).sort_values(["nll", "outcome_acc"], ascending=[True, False])
display(cls_report_df)
print("Classifier training minutes:", (time.time() - start) / 60)

outcome_model_names = list(outcome_val_proba_bank.keys())
outcome_val_proba = np.mean([outcome_val_proba_bank[name] for name in outcome_model_names], axis=0)
outcome_val_pred = outcome_val_proba.argmax(axis=1)
print("Avg classifier outcome_acc:", (outcome_val_pred == y_val_outcome).mean())
print("Avg classifier nll:", -np.log(np.clip(outcome_val_proba[np.arange(len(y_val_outcome)), y_val_outcome], 1e-12, 1)).mean())


Training outcome_cls_d6_seed42
0:	learn: 1.0830118	test: 1.0820564	best: 1.0820564 (0)	total: 19.4ms	remaining: 23.3s
150:	learn: 0.8786136	test: 0.8826689	best: 0.8826689 (150)	total: 1.49s	remaining: 10.4s
300:	learn: 0.8626315	test: 0.8788736	best: 0.8788736 (300)	total: 2.94s	remaining: 8.79s
450:	learn: 0.8523468	test: 0.8785197	best: 0.8784028 (399)	total: 4.33s	remaining: 7.19s
bestTest = 0.8784028455
bestIteration = 399
Shrink model to first 400 iterations.
{'classifier': 'outcome_cls_d6_seed42', 'outcome_acc': np.float64(0.5990733688753491), 'nll': np.float64(0.8784026939779004), 'best_iter': 399}

Training outcome_cls_d7_seed7
0:	learn: 1.0846235	test: 1.0838908	best: 1.0838908 (0)	total: 14.1ms	remaining: 14.1s
150:	learn: 0.8749492	test: 0.8810726	best: 0.8810726 (150)	total: 1.92s	remaining: 10.8s
300:	learn: 0.8579521	test: 0.8776855	best: 0.8776855 (300)	total: 3.8s	remaining: 8.81s
450:	learn: 0.8453870	test: 0.8768760	best: 0.8767613 (438)	total: 5.67s	remaining: 6.9s

,classifier,outcome_acc,nll,best_iter
1,outcome_cls_d7_seed7,0.599073,0.876761,438
2,outcome_cls_d5_seed99,0.598185,0.878190,586
0,outcome_cls_d6_seed42,0.599073,0.878403,399


Classifier training minutes: 0.34159650405248004
Avg classifier outcome_acc: 0.5990733688753491
Avg classifier nll: 0.8770902191489437


## 11. CatBoost Poisson lambda models

Model Poisson dipakai untuk menghasilkan `lambda_team` dan `lambda_opp`.

Nanti lambda ini di-blend dengan raw prediction regression ensemble, lalu dijadikan distribusi skor dengan bivariate Poisson.

In [12]:
POISSON_CONFIGS = [
    {
        "name": "pois_d6_seed42",
        "params": dict(loss_function="Poisson", eval_metric="Poisson", iterations=1400, learning_rate=0.040, depth=6, l2_leaf_reg=7, random_strength=1.0, bagging_temperature=0.5, random_seed=42),
    },
    {
        "name": "pois_d7_seed7",
        "params": dict(loss_function="Poisson", eval_metric="Poisson", iterations=1200, learning_rate=0.035, depth=7, l2_leaf_reg=9, random_strength=1.3, bagging_temperature=0.7, random_seed=7),
    },
    {
        "name": "pois_d5_seed99",
        "params": dict(loss_function="Poisson", eval_metric="Poisson", iterations=1600, learning_rate=0.030, depth=5, l2_leaf_reg=6, random_strength=1.8, bagging_temperature=0.9, random_seed=99),
    },
]

BASE_POIS_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)


def predict_poisson_lambda(model, X):
    # CatBoost Poisson uses an exponential link. Explicit Exponent prediction keeps lambda positive.
    try:
        pred = model.predict(X, prediction_type="Exponent")
    except TypeError:
        pred = model.predict(X)
        if np.nanmin(pred) <= 0:
            pred = np.exp(np.clip(pred, -7, 3))
    pred = np.asarray(pred, dtype=float)
    return np.clip(pred, 0.03, 8.0)


poisson_val_bank = {}
trained_val_poisson = {}
poisson_reports = []

start = time.time()
for cfg in POISSON_CONFIGS:
    name = cfg["name"]
    params = {**BASE_POIS_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training Poisson", name)

    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_tr, y_tr_team, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_team), use_best_model=True, early_stopping_rounds=150)
    mo.fit(X_tr, y_tr_opp, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_opp), use_best_model=True, early_stopping_rounds=150)

    lam_t = predict_poisson_lambda(mt, X_val)
    lam_o = predict_poisson_lambda(mo, X_val)
    poisson_val_bank[name] = (lam_t, lam_o)
    trained_val_poisson[name] = (mt, mo)

    pred_t, pred_o = postprocess_round_clip(lam_t, lam_o, max_score=MAX_SCORE)
    score, diag = compute_awmae(val_model, pred_t, pred_o)
    poisson_reports.append({"model": name, **diag, "best_iter_team": mt.best_iteration_, "best_iter_opp": mo.best_iteration_})
    print(poisson_reports[-1])

poisson_report_df = pd.DataFrame(poisson_reports).sort_values("AW-MAE")
display(poisson_report_df)
print("Poisson training minutes:", (time.time() - start) / 60)

poisson_model_names = list(poisson_val_bank.keys())
poisson_team_matrix = np.vstack([poisson_val_bank[name][0] for name in poisson_model_names])
poisson_opp_matrix = np.vstack([poisson_val_bank[name][1] for name in poisson_model_names])
poisson_val_team_lambda = np.mean(poisson_team_matrix, axis=0)
poisson_val_opp_lambda = np.mean(poisson_opp_matrix, axis=0)


Training Poisson pois_d6_seed42
0:	learn: 0.9765346	test: 0.9801894	best: 0.9801894 (0)	total: 31.1ms	remaining: 43.4s
150:	learn: 0.5819991	test: 0.6649647	best: 0.6649647 (150)	total: 4.36s	remaining: 36s
300:	learn: 0.5603131	test: 0.6564227	best: 0.6564227 (300)	total: 8.82s	remaining: 32.2s
450:	learn: 0.5467689	test: 0.6538341	best: 0.6537620 (434)	total: 13.1s	remaining: 27.7s
600:	learn: 0.5351664	test: 0.6521180	best: 0.6520927 (599)	total: 17.7s	remaining: 23.6s
750:	learn: 0.5256167	test: 0.6504932	best: 0.6504932 (750)	total: 22.2s	remaining: 19.2s
900:	learn: 0.5174400	test: 0.6496532	best: 0.6496432 (897)	total: 26.4s	remaining: 14.6s
1050:	learn: 0.5094761	test: 0.6486149	best: 0.6485574 (1047)	total: 30s	remaining: 9.96s
1200:	learn: 0.5025199	test: 0.6484719	best: 0.6484719 (1200)	total: 34.2s	remaining: 5.66s
1350:	learn: 0.4953581	test: 0.6479834	best: 0.6479691 (1330)	total: 38.8s	remaining: 1.41s
1399:	learn: 0.4932728	test: 0.6477540	best: 0.6477449 (1398)	total:

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
1,pois_d7_seed7,3.018516,1.018247,0.099327,0.571719,0.236164,1187,1163
0,pois_d6_seed42,3.028691,1.021420,0.099010,0.572163,0.236799,1398,1383
2,pois_d5_seed99,3.056467,1.029703,0.094377,0.570195,0.232737,1222,1585


Poisson training minutes: 3.63453510204951


## 12. Bivariate Poisson expected AW-MAE selector

Kita buat distribusi probabilitas untuk skor `0-0` sampai `max_score-max_score`.

Formula bivariate Poisson:

```text
team_goals = A + C
opp_goals  = B + C

A ~ Pois(lambda_team - lambda_shared)
B ~ Pois(lambda_opp  - lambda_shared)
C ~ Pois(lambda_shared)
```

Distribusi skor lalu dikalibrasi dengan:
- outcome classifier probability,
- empirical score prior dari train,
- expected AW-MAE loss matrix.

In [13]:
def poisson_factorials(max_score):
    facts = np.ones(max_score + 1, dtype=float)
    for i in range(1, max_score + 1):
        facts[i] = facts[i - 1] * i
    return facts


def bivariate_poisson_distribution(lambda_team, lambda_opp, shared_lambda=0.0, max_score=6):
    lambda_team = np.clip(np.asarray(lambda_team, dtype=float), 0.03, 8.0)
    lambda_opp = np.clip(np.asarray(lambda_opp, dtype=float), 0.03, 8.0)
    lam3 = np.minimum(float(shared_lambda), 0.80 * np.minimum(lambda_team, lambda_opp))
    lam1 = np.clip(lambda_team - lam3, 1e-8, None)
    lam2 = np.clip(lambda_opp - lam3, 1e-8, None)

    states = np.array([(i, j) for i in range(max_score + 1) for j in range(max_score + 1)], dtype=int)
    facts = poisson_factorials(max_score)
    probs = np.zeros((len(lambda_team), len(states)), dtype=float)
    exp_term = np.exp(-(lam1 + lam2 + lam3))

    for s, (x, y) in enumerate(states):
        total = np.zeros(len(lambda_team), dtype=float)
        for k in range(min(x, y) + 1):
            total += (
                (lam1 ** (x - k)) / facts[x - k]
                * (lam2 ** (y - k)) / facts[y - k]
                * (lam3 ** k) / facts[k]
            )
        probs[:, s] = exp_term * total

    probs = np.clip(probs, 1e-300, None)
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs, states


def state_outcome_class(states):
    diff = states[:, 0] - states[:, 1]
    return np.where(diff > 0, 2, np.where(diff < 0, 0, 1))


def build_score_pair_prior(df, max_score=6, smoothing=1.0):
    counts = np.full((max_score + 1, max_score + 1), smoothing, dtype=float)
    team = np.clip(df["team_goals"].round().astype(int).to_numpy(), 0, max_score)
    opp = np.clip(df["opp_goals"].round().astype(int).to_numpy(), 0, max_score)
    for tg, og in zip(team, opp):
        counts[tg, og] += 1.0
    return (counts / counts.sum()).reshape(-1)


def awmae_loss_matrix(states):
    # Rows are candidate predictions, columns are possible true scores.
    cand_team = states[:, 0][:, None]
    cand_opp = states[:, 1][:, None]
    true_team = states[:, 0][None, :]
    true_opp = states[:, 1][None, :]

    mae = (np.abs(true_team - cand_team) + np.abs(true_opp - cand_opp)) / 2
    exact = ((true_team == cand_team) & (true_opp == cand_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(cand_team, cand_opp)).astype(int)
    gd = ((true_team - true_opp) == (cand_team - cand_opp)).astype(int)

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    return ((mae + penalty) * multiplier) ** 1.5


def calibrate_score_distribution(probs, states, outcome_proba=None, classifier_power=0.0, score_prior=None, prior_power=0.0):
    probs = probs.copy()
    if outcome_proba is not None and classifier_power > 0:
        cls = state_outcome_class(states)
        probs *= np.clip(outcome_proba[:, cls], 1e-12, 1.0) ** classifier_power
    if score_prior is not None and prior_power > 0:
        probs *= np.clip(score_prior, 1e-12, 1.0)[None, :] ** prior_power
    probs = np.clip(probs, 1e-300, None)
    return probs / probs.sum(axis=1, keepdims=True)


def select_by_expected_loss(probs, states):
    loss = awmae_loss_matrix(states)
    expected = probs @ loss.T
    best_idx = np.argmin(expected, axis=1)
    return states[best_idx, 0].astype(int), states[best_idx, 1].astype(int)


blend_weights = best["weights"] / best["weights"].sum()
reg_val_team_raw = np.average(team_matrix, axis=0, weights=blend_weights)
reg_val_opp_raw = np.average(opp_matrix, axis=0, weights=blend_weights)

base_clip_rows = []
for max_score in range(4, 9):
    pred_t, pred_o = postprocess_round_clip(reg_val_team_raw, reg_val_opp_raw, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t, pred_o)
    base_clip_rows.append({"max_score": max_score, **diag})
base_clip_df = pd.DataFrame(base_clip_rows).sort_values("AW-MAE")
display(base_clip_df)

base_max_score = int(base_clip_df.iloc[0]["max_score"])
base_pred_t, base_pred_o = postprocess_round_clip(reg_val_team_raw, reg_val_opp_raw, max_score=base_max_score)
base_score, base_diag = compute_awmae(val_model, base_pred_t, base_pred_o)
print("Round+clip best:", base_score, "max_score:", base_max_score)

max_score_candidates = sorted(set([base_max_score, max(4, base_max_score - 1), min(8, base_max_score + 1)]))
lambda_blends = [0.00, 0.25, 0.50, 0.75, 1.00]  # 0=regression raw, 1=Poisson lambda
shared_lambdas = [0.00, 0.03, 0.06, 0.10, 0.15]
classifier_powers = [0.00, 0.50, 1.00, 1.75]
prior_powers = [0.00, 0.25, 0.50]

pp_rows = []
best_pp = {
    "score": base_score,
    "params": {
        "mode": "round_clip",
        "max_score": base_max_score,
        "lambda_blend": 0.0,
        "shared_lambda": 0.0,
        "classifier_power": 0.0,
        "prior_power": 0.0,
    },
    "diag": base_diag,
}

start = time.time()
for max_score in max_score_candidates:
    score_prior = build_score_pair_prior(tr_model, max_score=max_score, smoothing=1.0)
    for lambda_blend in lambda_blends:
        lam_team = np.clip((1 - lambda_blend) * reg_val_team_raw + lambda_blend * poisson_val_team_lambda, 0.03, 8.0)
        lam_opp = np.clip((1 - lambda_blend) * reg_val_opp_raw + lambda_blend * poisson_val_opp_lambda, 0.03, 8.0)
        for shared_lambda in shared_lambdas:
            raw_probs, states = bivariate_poisson_distribution(lam_team, lam_opp, shared_lambda=shared_lambda, max_score=max_score)
            for classifier_power in classifier_powers:
                for prior_power in prior_powers:
                    probs = calibrate_score_distribution(
                        raw_probs,
                        states,
                        outcome_proba=outcome_val_proba,
                        classifier_power=classifier_power,
                        score_prior=score_prior,
                        prior_power=prior_power,
                    )
                    pred_t, pred_o = select_by_expected_loss(probs, states)
                    score, diag = compute_awmae(val_model, pred_t, pred_o)
                    params = {
                        "mode": "bivar_poisson_expected_awmae",
                        "max_score": max_score,
                        "lambda_blend": lambda_blend,
                        "shared_lambda": shared_lambda,
                        "classifier_power": classifier_power,
                        "prior_power": prior_power,
                    }
                    pp_rows.append({**params, **diag})
                    if score < best_pp["score"]:
                        best_pp = {"score": score, "params": params.copy(), "diag": diag.copy()}

pp_df = pd.DataFrame(pp_rows).sort_values("AW-MAE").reset_index(drop=True)
pp_df.to_csv(POSTPROCESS_REPORT_PATH, index=False)

BEST_PP_PARAMS = best_pp["params"]
BEST_PP_SCORE = best_pp["score"]
BEST_MAX_SCORE = int(BEST_PP_PARAMS["max_score"])

print("Poisson/bivariate search minutes:", (time.time() - start) / 60)
print("Best score:", BEST_PP_SCORE)
print("Best params:", BEST_PP_PARAMS)
print("Best diagnostics:", best_pp["diag"])
display(pp_df.head(25))

,max_score,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,4,3.055714,1.008029,0.104659,0.54227,0.236799
1,5,3.061339,1.009235,0.102881,0.54227,0.234514
2,6,3.064800,1.010123,0.103707,0.54227,0.235339
3,7,3.069997,1.011900,0.102628,0.54227,0.234323
4,8,3.072564,1.012821,0.102628,0.54227,0.234323


Round+clip best: 3.055713990469004 max_score: 4
Poisson/bivariate search minutes: 0.1165490706761678
Best score: 2.9184656701776035
Best params: {'mode': 'bivar_poisson_expected_awmae', 'max_score': 4, 'lambda_blend': 1.0, 'shared_lambda': 0.15, 'classifier_power': 0.0, 'prior_power': 0.0}
Best diagnostics: {'AW-MAE': np.float64(2.9184656701776035), 'MAE_raw_component': np.float64(1.0064419903528814), 'exact_acc': np.float64(0.11430566133536431), 'outcome_acc': np.float64(0.5912668189895912), 'goal_diff_acc': np.float64(0.23794110180248795)}


,mode,max_score,lambda_blend,shared_lambda,classifier_power,prior_power,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,bivar_poisson_expected_awmae,4,1.00,0.15,0.0,0.00,2.918466,1.006442,0.114306,0.591267,0.237941
1,bivar_poisson_expected_awmae,4,1.00,0.06,0.0,0.00,2.918508,1.008917,0.114496,0.592473,0.239274
2,bivar_poisson_expected_awmae,5,1.00,0.15,0.0,0.00,2.918641,1.005077,0.112592,0.593552,0.235275
3,bivar_poisson_expected_awmae,5,0.50,0.10,0.0,0.00,2.919489,1.004221,0.112211,0.589490,0.237433
4,bivar_poisson_expected_awmae,5,0.75,0.15,0.0,0.00,2.920238,1.004189,0.113036,0.591394,0.236481
5,bivar_poisson_expected_awmae,4,0.75,0.15,0.0,0.00,2.920446,1.006474,0.114242,0.587649,0.239591
6,bivar_poisson_expected_awmae,5,0.75,0.06,0.0,0.00,2.920587,1.007394,0.112021,0.593234,0.236291
7,bivar_poisson_expected_awmae,4,0.75,0.10,0.0,0.00,2.920636,1.007521,0.114369,0.589109,0.239464
8,bivar_poisson_expected_awmae,5,1.00,0.10,0.0,0.00,2.921384,1.006791,0.111450,0.593615,0.234133
9,bivar_poisson_expected_awmae,4,1.00,0.10,0.0,0.00,2.921546,1.008346,0.114242,0.591648,0.238258


## 13. Train final models dan generate Poisson/Bivariate submission

In [14]:
final_pred_bank = {}
final_models = {}

for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    val_t, val_o = trained_val_models[name]
    best_iter = int(max(getattr(val_t, "best_iteration_", 800), getattr(val_o, "best_iteration_", 800)) + 140)
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 350)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final regression training", name, "iterations:", params["iterations"])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=CAT_FEATURE_INDICES, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=CAT_FEATURE_INDICES, verbose=150)
    final_pred_bank[name] = (np.clip(mt.predict(X_test), 0, None), np.clip(mo.predict(X_test), 0, None))
    final_models[name] = (mt, mo)

final_outcome_probas = []
final_classifiers = {}
for cfg in CLASSIFIER_CONFIGS:
    name = cfg["name"]
    val_clf = trained_val_classifiers[name]
    best_iter = int(getattr(val_clf, "best_iteration_", 700) + 120)
    params = {**BASE_CLS_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 250)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final classifier training", name, "iterations:", params["iterations"])
    clf = CatBoostClassifier(**params)
    clf.fit(X_full, y_full_outcome, cat_features=CAT_FEATURE_INDICES, verbose=150)
    final_outcome_probas.append(aligned_outcome_proba(clf, X_test))
    final_classifiers[name] = clf

test_outcome_proba = np.mean(final_outcome_probas, axis=0)

final_poisson_bank = {}
final_poisson_models = {}
for cfg in POISSON_CONFIGS:
    name = cfg["name"]
    val_t, val_o = trained_val_poisson[name]
    best_iter = int(max(getattr(val_t, "best_iteration_", 800), getattr(val_o, "best_iteration_", 800)) + 140)
    params = {**BASE_POIS_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 300)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final Poisson training", name, "iterations:", params["iterations"])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=CAT_FEATURE_INDICES, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=CAT_FEATURE_INDICES, verbose=150)
    final_poisson_bank[name] = (predict_poisson_lambda(mt, X_test), predict_poisson_lambda(mo, X_test))
    final_poisson_models[name] = (mt, mo)

test_team_matrix = np.vstack([final_pred_bank[name][0] for name in model_names])
test_opp_matrix = np.vstack([final_pred_bank[name][1] for name in model_names])
weights = best["weights"] / best["weights"].sum()
reg_test_team_raw = np.average(test_team_matrix, axis=0, weights=weights)
reg_test_opp_raw = np.average(test_opp_matrix, axis=0, weights=weights)

poisson_test_team_matrix = np.vstack([final_poisson_bank[name][0] for name in poisson_model_names])
poisson_test_opp_matrix = np.vstack([final_poisson_bank[name][1] for name in poisson_model_names])
poisson_test_team_lambda = np.mean(poisson_test_team_matrix, axis=0)
poisson_test_opp_lambda = np.mean(poisson_test_opp_matrix, axis=0)

round_team, round_opp = postprocess_round_clip(reg_test_team_raw, reg_test_opp_raw, max_score=BEST_MAX_SCORE)
submission_round = sample[["Id"]].copy()
submission_round["team_goals"] = round_team
submission_round["opp_goals"] = round_opp
submission_round.to_csv(SUBMISSION_ROUNDCLIP_PATH, index=False)

if BEST_PP_PARAMS.get("mode") == "bivar_poisson_expected_awmae":
    lambda_blend = BEST_PP_PARAMS["lambda_blend"]
    max_score = int(BEST_PP_PARAMS["max_score"])
    lam_team = np.clip((1 - lambda_blend) * reg_test_team_raw + lambda_blend * poisson_test_team_lambda, 0.03, 8.0)
    lam_opp = np.clip((1 - lambda_blend) * reg_test_opp_raw + lambda_blend * poisson_test_opp_lambda, 0.03, 8.0)
    probs, states = bivariate_poisson_distribution(lam_team, lam_opp, shared_lambda=BEST_PP_PARAMS["shared_lambda"], max_score=max_score)
    score_prior = build_score_pair_prior(full_train_model, max_score=max_score, smoothing=1.0)
    probs = calibrate_score_distribution(
        probs,
        states,
        outcome_proba=test_outcome_proba,
        classifier_power=BEST_PP_PARAMS["classifier_power"],
        score_prior=score_prior,
        prior_power=BEST_PP_PARAMS["prior_power"],
    )
    final_team, final_opp = select_by_expected_loss(probs, states)
    active_mode = "bivar_poisson_expected_awmae"
else:
    final_team, final_opp = round_team, round_opp
    active_mode = "round_clip"

submission = sample[["Id"]].copy()
submission["team_goals"] = final_team
submission["opp_goals"] = final_opp
assert submission.shape == sample.shape
assert submission["Id"].equals(sample["Id"])
submission.to_csv(SUBMISSION_PATH, index=False)

changed_rows = ((submission["team_goals"] != submission_round["team_goals"]) | (submission["opp_goals"] != submission_round["opp_goals"])).sum()

print("Saved submission:", SUBMISSION_PATH)
print("Saved round+clip backup:", SUBMISSION_ROUNDCLIP_PATH)
print("Active mode:", active_mode)
print("Validation best regression blend AW-MAE:", best["score"])
print("Validation best bivariate/Poisson AW-MAE:", BEST_PP_SCORE)
print("Best params:", BEST_PP_PARAMS)
print("Changed rows vs regression round+clip:", changed_rows)
print("Shape:", submission.shape)

display(submission.head())
display(submission[["team_goals", "opp_goals"]].describe())

print("\ndistribusi skor")
print("\nteam_goals")
print(submission["team_goals"].value_counts())
print("\nopp_goals")
print(submission["opp_goals"].value_counts())

print("\ndistribusi pasangan skor")
display(submission[["team_goals", "opp_goals"]].value_counts().head(30).to_frame("count"))


Final regression training recon_cat_mae_d7_seed42 iterations: 1739


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1661994	total: 18.9ms	remaining: 32.9s
150:	learn: 1.0657904	total: 2.76s	remaining: 29s
300:	learn: 1.0346089	total: 5.48s	remaining: 26.2s
450:	learn: 1.0206316	total: 8.32s	remaining: 23.8s
600:	learn: 1.0121012	total: 11.1s	remaining: 21s
750:	learn: 1.0061714	total: 13.8s	remaining: 18.2s
900:	learn: 1.0005557	total: 16.6s	remaining: 15.4s
1050:	learn: 0.9953752	total: 19.3s	remaining: 12.6s
1200:	learn: 0.9907992	total: 22.1s	remaining: 9.9s
1350:	learn: 0.9869910	total: 24.9s	remaining: 7.14s
1500:	learn: 0.9830708	total: 27.6s	remaining: 4.38s
1650:	learn: 0.9792976	total: 30.4s	remaining: 1.62s
1738:	learn: 0.9774627	total: 32s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1662431	total: 16.6ms	remaining: 28.9s
150:	learn: 1.0658665	total: 2.64s	remaining: 27.8s
300:	learn: 1.0347086	total: 5.28s	remaining: 25.2s
450:	learn: 1.0210968	total: 7.96s	remaining: 22.7s
600:	learn: 1.0122467	total: 10.7s	remaining: 20.2s
750:	learn: 1.0064112	total: 13.3s	remaining: 17.6s
900:	learn: 1.0003985	total: 16.1s	remaining: 14.9s
1050:	learn: 0.9952989	total: 18.9s	remaining: 12.3s
1200:	learn: 0.9908904	total: 21.6s	remaining: 9.66s
1350:	learn: 0.9872928	total: 24.3s	remaining: 6.99s
1500:	learn: 0.9834157	total: 27.1s	remaining: 4.3s
1650:	learn: 0.9798721	total: 30.4s	remaining: 1.62s
1738:	learn: 0.9778793	total: 32.3s	remaining: 0us

Final regression training recon_cat_mae_d6_seed7 iterations: 1933


...

0:	learn: 1.1661218	total: 22.1ms	remaining: 24.6s
150:	learn: 1.0700301	total: 3.36s	remaining: 21.4s
300:	learn: 1.0353580	total: 6.68s	remaining: 18s
450:	learn: 1.0196145	total: 10s	remaining: 14.7s
600:	learn: 1.0101054	total: 13.4s	remaining: 11.3s
750:	learn: 1.0032033	total: 16.7s	remaining: 8.01s
900:	learn: 0.9977478	total: 20.1s	remaining: 4.68s
1050:	learn: 0.9919985	total: 23.5s	remaining: 1.34s
1110:	learn: 0.9897929	total: 24.9s	remaining: 0us

Final regression training recon_cat_rmse_d7_seed123 iterations: 991
0:	learn: 1.7756212	total: 17.1ms	remaining: 16.9s
150:	learn: 1.4389151	total: 2.81s	remaining: 15.6s
300:	learn: 1.4024274	total: 5.61s	remaining: 12.9s
450:	learn: 1.3775230	total: 8.42s	remaining: 10.1s
600:	learn: 1.3575928	total: 11.3s	remaining: 7.3s
750:	learn: 1.3416695	total: 14.1s	remaining: 4.5s
900:	learn: 1.3259810	total: 17s	remaining: 1.69s
990:	learn: 1.3177300	total: 18.6s	remaining: 0us
0:	learn: 1.7754276	total: 17ms	remaining: 16.8s
150:	learn

,Id,team_goals,opp_goals
0,M034984_Seychelles,2,1
1,M034984_Mauritius,1,2
2,M034985_Comoros,1,2
3,M034985_Maldives,2,1
4,M034986_Réunion,2,1


,team_goals,opp_goals
count,42422.000000,42422.000000
mean,1.198340,1.198859
std,0.848779,0.850461
min,0.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,4.000000,4.000000



distribusi skor

team_goals
team_goals
1    19699
2    11373
0     8646
3     2425
4      279
Name: count, dtype: int64

opp_goals
opp_goals
1    19511
2    11461
0     8727
3     2467
4      256
Name: count, dtype: int64

distribusi pasangan skor


count
team_goals opp_goals       
1          2           8455
2          1           8314
1          1           7256
           0           3544
0          1           3457
2          0           2994
0          2           2942
           3           1994
3          0           1923
           1            471
1          3            441
4          0            266
0          4            253
2          2             33
           3             32
3          2             31
4          1             13
1          4              3

## 14. Notes

Kalau hasil Kaggle naik, berarti probabilistic score distribution membantu exact score/GD.

Kalau Kaggle turun:
1. bandingkan dengan Percobaan 7 classifier-aware,
2. coba `lambda_blend` yang lebih kecil secara manual,
3. kurangi `classifier_power` kalau distribusi terlalu ekstrim,
4. next upgrade: pseudo-sequential test update atau Dixon-Coles low-score correction eksplisit.